# Datasheet Analysis
we read the excel datasheet and try to understand its content whilst paying attention to any possible anomalies

In [ ]:
import pandas as pd

# We read the given excel datasheet
df = pd.read_excel("Quality_Data_v3.xlsx", engine="openpyxl")

In [7]:
# We dont want to skip rows during display so we set the following options
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

### Basic datasheet info
We begin by checking basic datasheet info such as number of rows, columns, missing values and duplicated rows

In [9]:
print("Datasheet Rows:", len(df))
print("Datasheet Columns:", len(df.columns))

Rows: 8198
Columns: 23


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8198 entries, 0 to 8197
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Defect ID                8198 non-null   object 
 1   Report Date              8198 non-null   object 
 2   Report Time              8198 non-null   object 
 3   Production Date          8198 non-null   object 
 4   VIN                      8198 non-null   object 
 5   Car Model                8198 non-null   object 
 6   Motor Type               8198 non-null   object 
 7   Design Package           8198 non-null   object 
 8   Station ID               8198 non-null   object 
 9   Station Name             8198 non-null   object 
 10  Part Number              8198 non-null   int64  
 11  Part Name                8198 non-null   object 
 12  Supplier                 8198 non-null   object 
 13  Defect Name              8198 non-null   object 
 14  Defect Category         

In [12]:
print("Duplicate rows:", df.duplicated().sum()) # We check if any exact same rows were inserted twice
print("Duplicate Defect IDs:", df["Defect ID"].duplicated().sum()) # We check if any rows has the same Defect IDs

Duplicate rows: 0
Duplicate Defect IDs: 0


## Conflicting data
We investigate different defect reports and find some VIN (Vehicle Identification Number) were reported with different Car model, Motor type and design package. We cannot proceed and delete such information because it's not feasible for us to find out which report is correct.

In [13]:
vehicle_columns = [
    "Car Model",
    "Motor Type",
    "Design Package"
]

conflicts = df.groupby("VIN")[vehicle_columns].nunique()

conflicting_vins = conflicts[
    (conflicts > 1).any(axis=1)
]

print("VINs with conflicting details:", len(conflicting_vins))
conflicting_vins.head()

VINs with conflicting details: 517


,Car Model,Motor Type,Design Package
VIN,,,
WBAX00000069,1,2,2
WBAX00000084,2,2,1
WBAX00000160,2,2,3
WBAX00000203,2,2,1
WBAX00000221,2,2,1


## Missing values
We double check for missing values but the only columns with missing values were the "Root Cause Identified","Resolution Timestamp", "Resolution Time (hours)","Rework Time (minutes)". The BMW presentation explicitly mentions those aren't missing data, it only means the defect hasn't been resolved yet. So we take no action here

In [14]:
missing_values = df.isna().sum()

display(missing_values[missing_values > 0].sort_values(ascending=False)) # The 4 rows with missing values aren't really missing values, they are just defects that haven't been resolved.

Root Cause Identified      1695
Resolution Timestamp       1251
Resolution Time (hours)    1251
Rework Time (minutes)      1251
dtype: int64

## Duplicate reports
In the following section we checked for repeated combinations of VIN and Defect Name to identify potentially duplicated defect reports. 
These records were flagged for inspection rather than automatically removed, as repeated reports may represent genuine recurring defects.

In [15]:
duplicate_mask = df.duplicated(
    subset=["VIN", "Defect Name"],
    keep=False
)

duplicate_defects = df[duplicate_mask].sort_values(
    by=["VIN", "Defect Name"]
)

print("Rows with repeated VIN and Defect Name:", len(duplicate_defects))

display(duplicate_defects)

Rows with repeated VIN and Defect Name: 116


,Defect ID,Report Date,Report Time,Production Date,VIN,Car Model,Motor Type,Design Package,Station ID,Station Name,Part Number,Part Name,Supplier,Defect Name,Defect Category,Severity Rating,Inspector ID,Production Shift,Resolution Status,Resolution Timestamp,Resolution Time (hours),Root Cause Identified,Rework Time (minutes)
210,DEF-2026-000211,2026-01-05,17:54,2026-01-05,WBAX00001327,Alpina,Long Range,Luxury,ST-09,EV Battery Installation,400415,Battery Housing,Voltania Cells,Coolant Leak,Functional,4,INS-013,Afternoon,Resolved,2026-01-06 03:35,9.69,Yes,53.0
211,DEF-2026-000212,2026-01-05,18:05,2026-01-05,WBAX00001327,Alpina,Long Range,Luxury,ST-09,EV Battery Installation,400415,Battery Housing,Voltania Cells,Coolant Leak,Functional,4,INS-022,Afternoon,Resolved,2026-01-06 03:35,9.69,Yes,53.0
330,DEF-2026-000331,2026-01-08,06:23,2026-01-08,WBAX00002118,Base,Long Range,Luxury,ST-13,Windshield Installation,400149,Body Panel,Hoffmann Metallwerk,Paint Run,Cosmetic,9,INS-022,Morning,Resolved,2026-01-08 14:14,7.86,NaN,60.0
400,DEF-2026-000401,2026-01-09,12:50,2026-01-08,WBAX00002118,Base,High Performance,Luxury,ST-01,Body Shop Framing,400149,Body Panel,Hoffmann Metallwerk,Paint Run,Cosmetic,8,INS-039,Morning,Resolved,2026-01-09 17:21,4.53,No,48.0
537,DEF-2026-000538,2026-01-13,11:33,2026-01-13,WBAX00003632,Base,High Performance,Luxury,ST-17,Charging System Inspection,400344,Harness Connector,Kessler-Automotive,Connector Corrosion,Electrical,4,INS-025,Morning,In Progress,NaN,NaN,NaN,NaN
707,DEF-2026-000708,2026-01-16,21:54,2026-01-13,WBAX00003632,Long,Long Range,Race,ST-08,Wire Harness Installation,400330,Harness Connector,Kessler-Automotive,Connector Corrosion,Electrical,4,INS-009,Afternoon,In Progress,NaN,NaN,NaN,NaN
675,DEF-2026-000676,2026-01-16,06:58,2026-01-16,WBAX00004640,Pick-Up,High Performance,Race,ST-01,Body Shop Framing,400118,Body Panel,Hoffmann Metallwerk,Paint Run,Cosmetic,9,INS-001,Morning,Resolved,2026-01-16 19:42,12.74,Yes,73.0
692,DEF-2026-000693,2026-01-16,14:22,2026-01-16,WBAX00004640,iX0M,Long Range,Luxury,ST-04,Paint Shop Topcoat,400161,Body Panel,Hoffmann Metallwerk,Paint Run,Cosmetic,10,INS-031,Afternoon,Resolved,2026-01-17 00:25,10.06,Yes,12.0
1361,DEF-2026-001362,2026-02-02,01:46,2026-02-02,WBAX00009803,Long,Long Range,Offroad,ST-08,Wire Harness Installation,400458,Control Unit,Nordwerk Systems,Sensor Failure,Electrical,4,INS-017,Night,Resolved,2026-02-02 12:05,10.32,Yes,122.0
1384,DEF-2026-001385,2026-02-02,13:55,2026-02-02,WBAX00009803,Long,High Performance,Race,ST-14,Brake System Assembly,400523,Sensor Module,Nordwerk Systems,Sensor Failure,Electrical,3,INS-040,Morning,Resolved,2026-02-02 16:43,2.80,Yes,138.0


In [16]:
# Count repeated defect names for each VIN
duplicate_counts = (
    df.groupby(["VIN", "Defect Name"])
    .size()
    .reset_index(name="Report Count")
    .query("`Report Count` > 1")
    .sort_values("Report Count", ascending=False)
)

print("Number of repeated VIN–Defect combinations:", len(duplicate_counts))

display(duplicate_counts)

Number of repeated VIN–Defect combinations: 58


,VIN,Defect Name,Report Count
215,WBAX00001327,Coolant Leak,2
7038,WBAX00047395,Frame Misalignment,2
5886,WBAX00040139,Sensor Failure,2
5953,WBAX00040662,Colour Mismatch,2
5983,WBAX00040900,Frame Misalignment,2
6058,WBAX00041333,Frame Misalignment,2
6123,WBAX00041670,Orange Peel Finish,2
6295,WBAX00042812,Trim Misfit,2
6336,WBAX00043110,Weld Seam Crack,2
6418,WBAX00043579,Frame Misalignment,2
